# Complete YOLO + VLM Orchestration Pipeline

This notebook runs the full pipeline on a sample image: locates the placard using YOLOv11 and reads the hazard codes using a VLM model.

In [ ]:
import sys
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path().resolve().parent / "src"))

In [ ]:
from un_detector.pipeline.main_pipeline import UNNumberPipeline

# Define paths for dataset and model outputs
yolo_model_path = "../data/yolo/yolo11x_earlystopping.pt"
un_labels_csv = "../kaggle_dataset/un-number-labels.csv"

# Initialize the pipeline (using standard Tesseract/EasyOCR/Idefics2 engines)
# EasyOCR is used here as a fast default, swap to 'idefics2' for the high-end VLM
pipeline = UNNumberPipeline(
    yolo_model_path=yolo_model_path,
    ocr_engine_type="easyocr",
    un_labels_csv=un_labels_csv
)

## Run Pipeline on Test Image

In [ ]:
test_image_path = "../images/hazard_plate.jpg"

if Path(test_image_path).exists():
    # Run detection and OCR OCR on the image
    predictions = pipeline.run(test_image_path, conf_threshold=0.25)
    
    print(f"Predictions found: {len(predictions)}")
    for i, pred in enumerate(predictions):
        print(f"Placard #{i+1}:")
        print(f"  - Bounding Box: {pred['bbox']}")
        print(f"  - HIN (Upper):  {pred['hin_number']}")
        print(f"  - UN (Lower):   {pred['un_number']}")
        print(f"  - Description:  {pred['description']}")
        
    # Visualize predictions
    pipeline.visualize_predictions(test_image_path, predictions)
else:
    print(f"Image not found at {test_image_path}, please specify a valid image path.")